# Multilevel and Marginal Linear Regression

---

## A Case Study with the HSB Dataset

This notebook serves as a practical application of the concepts discussed in Week 3, focusing on multilevel and marginal models for dependent data. We will use the "High School and Beyond" (HSB) dataset, which has a natural clustured structure, to explore these advanced regression techniques.

Note that some of the models illustrated in this notebook may take a few moments to fit.

We begin by importing the libraries that we will be using.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

Data can be **dependent**, or have a **multilevel structure**, for many reasons. A common source of dependence is **cluster sampling**, where observations are collected in groups. In this notebook, we will analyze the data from students who are grouped, or "nested", within different schools.

First, we'll load the HSB dataset. This dataset is [available online](https://nces.ed.gov/surveys/hsb/), so we can read it directly into a Pandas dataframe.

In [7]:
# Load the High School and Beyond (HSB) dataset from an online source
try:
    da = pd.read_csv("https://stats.idre.ucla.edu/stat/data/hsbdemo.csv")
except Exception as e:
    print(f"Failed to load data. Error: {e}")
    print("Please check your internet connection or the URL.")
    # As a fallback, create a dummy dataframe to avoid further errors
    da = pd.DataFrame({'id': [], 'female': [], 'ses': [], 'schtyp': [], 'prog': [], 'read': [], 'write': [], 'math': [], 'science': [], 'socst': [], 'honors': [], 'awards': [], 'cid': []})
    
# Rename columns for clarity and consistency
da = da.rename(columns={"id": "student_id", "cid": "school_id"})

# Display the first few rows to inspect the data
da.head()

,student_id,female,ses,schtyp,prog,read,write,math,science,socst,honors,awards,school_id
0,45,female,low,public,vocation,34,35,41,29,26,not enrolled,0,1
1,108,male,middle,public,general,34,33,41,36,36,not enrolled,0,1
2,15,male,high,public,vocation,39,39,44,26,42,not enrolled,0,1
3,67,male,low,public,vocation,37,37,42,33,32,not enrolled,0,1
4,153,male,middle,public,vocation,39,31,40,39,51,not enrolled,0,1



---

## Introduction to Clustered Data
The HSB dataset is a classic example of clustered data. The observations are students, but these students are grouped within schools. Students within the same schools share teachers, resources, curriculum, and a common social environment. This means the math score of two students from the same school is likely to be more similar than the achievement of two students from different schools. This violates the independence assumption of standard regression models.

### Clustering Structure in the HSB Data
The `school_id` variable represents the cluster identifier. We will use this variable to account for the dependency in our models.

---

## Intraclass Correlation Coefficient (ICC)
The **Intraclass Correlation Coefficient (ICC)** measures the similarity of observations within the same cluster. It ranges from 0 (no similarity, complete independence) to 1 (perfect similarity, all observations in a cluster are identical).

We can estimate the ICC using **Generalized Estimating Equations (GEE)**, a technique for fitting marginal models. We'll start by calculating the ICC for our main outcome variable, `math`.